In [68]:
from typing import Optional
from enum import Enum

from dataclasses import dataclass

from ping3 import ping
import netifaces
import speedtest

In [72]:
@dataclass 
class PingMetric:
    value: float

    
@dataclass
class IPAddress:
    value: str

@dataclass
class SpeedMetric:
    download: float
    upload: float
    ping: float

class Units(Enum):
    SECONDS: str = "seconds"
    MBPS: str = "Mbps"
        
@dataclass
class Metric:
    value: float
    unit: Optional[Units] = None
    
    


In [74]:
Metric(value=10, unit="seconds")

Metric(value=10, unit='seconds')

In [66]:
def get_default_gateway():
    return netifaces.gateways()['default'][netifaces.AF_INET][0]

def do_ping(address: IPAddress, num_pings: int) -> Metric:
    pings = [ping(address.value) for _ in range(num_pings)]
    avg_ping = sum(pings) / len(pings)
    return Metric(
        value=avg_ping,
        unit="seconds",
        )

def do_speed_test() -> SpeedMetric:
    s = speedtest.Speedtest()
    s.download()
    s.upload()
    results = s.results.dict()
    return SpeedMetric(
        download=results.get("download"), # Mbps
        upload=results.get("upload"), # Mbps
        ping=results.get("ping") # ms
    )
    
speed_test = do_speed_test()

download_speed = Metric(value=speed_test.download, unit="Mbps")
upload_speed = Metric(value=speed_test.upload, unit="Mbps")

In [62]:
LOCAL_ROUTER = IPAddress("192.168.0.1")
GOOGLE_DNS = IPAddress("8.8.8.8")
CLOUDFLARE = IPAddress("1.1.1.1")

def main():
    num_pings = 50
    local_router = do_ping(LOCAL_ROUTER, num_pings)
    google_dns = do_ping(GOOGLE_DNS, num_pings)
    cloudflare = do_ping(CLOUDFLARE, num_pings)
    
    print(f"{local_router=}")
    print(f"{google_dns=}")
    print(f"{cloudflare=}")
    
    speed_test_results = do_speed_test()
    
    print(f"{speed_test_results=}")

main()

local_router=PingMetric(value=0.009646921157836915)
google_dns=PingMetric(value=0.021892070770263672)
cloudflare=PingMetric(value=0.021999893188476564)
speed_test_results=SpeedMetric(download=8025717.293217934, upload=8724022.04283887, ping=14.172)


In [57]:
speed_test_results

SpeedMetric(download=1967453.3379144818, upload=7239192.583292722, ping=27.795)

# 2. Wi-Fi Health (if applicable)

## Signal Strength
- Use iwconfig or iw on Linux to get RSSI and link quality.
- Show this per device or across rooms (if you're doing localized scans).

## Channel Congestion
- Scan neighboring SSIDs to detect:
- Channel overlap (e.g., 2.4GHz congestion)
- Signal-to-noise ratio (SNR)

## Bandwidth Per Device
- Use nload, iftop, or vnstat to track interface traffic.
-You could even parse your router’s device list (some expose this via API or scrapeable status page).